In [6]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import lfilter

In [7]:
# Load audio
y, sr = librosa.load("../raw_audio/dummy.wav", sr=None)

In [11]:
frame_length = int(0.03 * sr)
hop_length = frame_length // 2
order = 16

y = np.pad(y, (0, frame_length), mode="constant")

frames = librosa.util.frame(
    y,
    frame_length=frame_length,
    hop_length=hop_length
)

window = np.hanning(frame_length)

output = np.zeros(len(y))
norm = np.zeros(len(y))

# keep previous coefficients for optional smooth drift
prev_a = None

# jitter strength (small = stable, larger = more chaotic)
jitter_strength = 0.05

for i in range(frames.shape[1]):

    frame = frames[:, i] * window

    # LPC analysis
    a = librosa.lpc(frame, order=order)

    # ---------------------------
    # BOUNDED COEFFICIENT JITTER
    # ---------------------------

    noise = np.random.randn(len(a))
    noise = np.convolve(noise, [1, 2, 3, 2, 1], mode="same") / 9
    a += 0.002 * noise

    # enforce LPC constraint (important)
    a[0] = 1.0

    # optional: stabilize drift by mild smoothing across time
    # if prev_a is not None:
    #     a = 0.7 * a + 0.3 * prev_a

    prev_a = a

    # residual (excitation)
    residual = lfilter(a, [1.0], frame)

    # synthesis
    reconstructed = lfilter([1.0], a, residual)

    start = i * hop_length

    output[start:start + frame_length] += reconstructed * window
    norm[start:start + frame_length] += window**2

# normalize overlap-add
output /= (norm + 1e-8)

sf.write("../processed_audio/lpc_jitter.wav", output, sr)